<a href="https://colab.research.google.com/github/turkestanova2006/ai-2026-hw2/blob/main/HW3%20Turkestanova%20Aida.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# HW3 — Two retrievers and a model: what RAG actually sends

**Name:**Turkestanova Aida
**Student ID:**s23070029
**Group:**
**Repository:**https://github.com/turkestanova2006/css4007-hw3

## AI tool disclosure

State which AI tools you used and for what. Expected and fine; undisclosed use is not.

>

Run this notebook **top to bottom** before you push. A cell with no saved output earns nothing.
The instructions for every section are in `README.md`.

---
# Part A — relevance by hand (1 pt)

| | Text |
|---|---|
| D1 | Large language models can generate text and answer questions. |
| D2 | Retrieval-Augmented Generation retrieves relevant documents before generating an answer. |
| D3 | Embeddings represent text as numerical vectors. |
| Query | How does RAG retrieve information? |

`k1 = 1.5`, `b = 0.75`, `idf(t) = ln((N − n(t) + 0.5) / (n(t) + 0.5))`. Tokenise as README §3 says.

## A1 (a) — BM25 as written

We use lowercase words, split by spaces and hyphens, and remove punctuation. No stemming or stop-word removal is used.

**Document tokens and lengths:**

| Doc | Tokens | Length |
|---|---|---:|
| D1 | large, language, models, can, generate, text, and, answer, questions | 9 |
| D2 | retrieval, augmented, generation, retrieves, relevant, documents, before, generating, an, answer | 10 |
| D3 | embeddings, represent, text, as, numerical, vectors | 6 |

**Average document length:**

$$
avgdl=\frac{9+10+6}{3}=\frac{25}{3}=8.3333
$$

**Query terms:**

`how, does, rag, retrieve, information`

**Term Frequency (TF), Document Frequency (DF), and IDF:**

| Query term | D1 TF | D2 TF | D3 TF | DF | IDF |
|---|---:|---:|---:|---:|---:|
| how | 0 | 0 | 0 | 0 | 1.9459 |
| does | 0 | 0 | 0 | 0 | 1.9459 |
| rag | 0 | 0 | 0 | 0 | 1.9459 |
| retrieve | 0 | 0 | 0 | 0 | 1.9459 |
| information | 0 | 0 | 0 | 0 | 1.9459 |

There are 3 documents, so $N=3$.

For every query term, $DF=0$.

**IDF calculation:**

$$
IDF(t)=\ln\left(\frac{N-DF+0.5}{DF+0.5}\right)
$$

$$
IDF(t)=\ln\left(\frac{3-0+0.5}{0+0.5}\right)
$$

$$
IDF(t)=\ln(7)=1.9459
$$

**BM25 calculation:**

Parameters:

$$
k_1=1.5,\quad b=0.75
$$

$$
BM25(q,d)=\sum_{t\in q}IDF(t)
\frac{f(t,d)(k_1+1)}
{f(t,d)+k_1(1-b+b\frac{|d|}{avgdl})}
$$

For D1, D2 and D3, every query term has TF = 0.

For example, for "retrieve" in D2:

$$
1.9459\times\frac{0(1.5+1)}
{0+1.5(1-0.75+0.75\frac{10}{8.3333})}=0
$$

Therefore:

$$
BM25(D1)=0+0+0+0+0=0
$$

$$
BM25(D2)=0+0+0+0+0=0
$$

$$
BM25(D3)=0+0+0+0+0=0
$$

**Final ranking:**

| Document | BM25 Score | Rank |
|---|---:|---|
| D1 | 0.0000 | Tied |
| D2 | 0.0000 | Tied |
| D3 | 0.0000 | Tied |

**Explanation:**

All documents have a BM25 score of zero because no query word exactly matches a document word. D2 contains "retrieval" and "retrieves", but not "retrieve". Without stemming, these words are treated as different tokens. There is no unique first document.


## A1 (b) — BM25 with stemming

In this part, we apply the given stemming rules.

**Stemming rules:**

| Original words | Stem |
|---|---|
| retrieve, retrieves, retrieval | retriev |
| generate, generating, generation | gener |
| information | inform |
| does | doe |

All other words remain unchanged.

**Stemmed document tokens:**

| Doc | Tokens | Length |
|---|---|---:|
| D1 | large, language, models, can, gener, text, and, answer, questions | 9 |
| D2 | retriev, augmented, gener, retriev, relevant, documents, before, gener, an, answer | 10 |
| D3 | embeddings, represent, text, as, numerical, vectors | 6 |

**Average document length:**

$$
avgdl=\frac{9+10+6}{3}=8.3333
$$

**Stemmed query:**

`how, doe, rag, retriev, inform`

**TF, DF and IDF table:**

| Query term | D1 TF | D2 TF | D3 TF | DF | IDF |
|---|---:|---:|---:|---:|---:|
| how | 0 | 0 | 0 | 0 | 1.9459 |
| doe | 0 | 0 | 0 | 0 | 1.9459 |
| rag | 0 | 0 | 0 | 0 | 1.9459 |
| retriev | 0 | 2 | 0 | 1 | 0.5108 |
| inform | 0 | 0 | 0 | 0 | 1.9459 |

The stem "retriev" appears twice in D2 and does not appear in D1 or D3.

Therefore:

$$
TF(retriev,D2)=2
$$

$$
DF(retriev)=1
$$

**IDF for "retriev":**

$$
IDF=\ln\left(\frac{3-1+0.5}{1+0.5}\right)
$$

$$
IDF=\ln\left(\frac{2.5}{1.5}\right)=0.5108
$$

For the other query terms, DF = 0, so:

$$
IDF=\ln(7)=1.9459
$$

**BM25 calculation:**

For D1, all query term frequencies are zero:

$$
BM25(D1)=0+0+0+0+0=0
$$

For D2, only "retriev" contributes to the score:

$$
BM25(D2)=0.5108\times
\frac{2(1.5+1)}
{2+1.5(1-0.75+0.75\frac{10}{8.3333})}
$$

$$
=0.5108\times\frac{5}{3.725}
$$

$$
BM25(D2)=0.6857
$$

For D3, all query term frequencies are zero:

$$
BM25(D3)=0+0+0+0+0=0
$$

**Final ranking:**

| Document | BM25 Score | Rank |
|---|---:|---:|
| D2 | 0.6857 | 1 |
| D1 | 0.0000 | Tied 2 |
| D3 | 0.0000 | Tied 2 |

**Comparison with A1(a):**

Without stemming, all documents have a BM25 score of zero. There is no unique first document.

With stemming, D2 ranks first because "retrieval" and "retrieves" are converted to "retriev", which matches the stemmed query.

Stemming improves matching between different forms of the same word.


## A2 — Cosine Similarity

**Given vectors:**

- Query = [0.8, 0.2, 0.5]
- D1 = [0.7, 0.1, 0.4]
- D2 = [0.9, 0.3, 0.6]
- D3 = [0.1, 0.8, 0.2]

**Cosine similarity formula:**

$$
cos(Q,D)=\frac{Q\cdot D}{\|Q\|\|D\|}
$$

### Step 1. Query magnitude

$$
\|Q\|=\sqrt{0.8^2+0.2^2+0.5^2}
$$

$$
=\sqrt{0.64+0.04+0.25}
$$

$$
=\sqrt{0.93}=0.9644
$$

### Step 2. Document D1

**Dot product:**

$$
Q\cdot D1=0.8(0.7)+0.2(0.1)+0.5(0.4)
$$

$$
=0.56+0.02+0.20=0.78
$$

**Magnitude:**

$$
\|D1\|=\sqrt{0.7^2+0.1^2+0.4^2}
$$

$$
=\sqrt{0.66}=0.8124
$$

**Cosine similarity:**

$$
cos(Q,D1)=\frac{0.78}{\sqrt{0.93}\sqrt{0.66}}
$$

$$
cos(Q,D1)=0.9956
$$

### Step 3. Document D2

**Dot product:**

$$
Q\cdot D2=0.8(0.9)+0.2(0.3)+0.5(0.6)
$$

$$
=0.72+0.06+0.30=1.08
$$

**Magnitude:**

$$
\|D2\|=\sqrt{0.9^2+0.3^2+0.6^2}
$$

$$
=\sqrt{1.26}=1.1225
$$

**Cosine similarity:**

$$
cos(Q,D2)=\frac{1.08}{\sqrt{0.93}\sqrt{1.26}}
$$

$$
cos(Q,D2)=0.9977
$$

### Step 4. Document D3

**Dot product:**

$$
Q\cdot D3=0.8(0.1)+0.2(0.8)+0.5(0.2)
$$

$$
=0.08+0.16+0.10=0.34
$$

**Magnitude:**

$$
\|D3\|=\sqrt{0.1^2+0.8^2+0.2^2}
$$

$$
=\sqrt{0.69}=0.8307
$$

**Cosine similarity:**

$$
cos(Q,D3)=\frac{0.34}{\sqrt{0.93}\sqrt{0.69}}
$$

$$
cos(Q,D3)=0.4244
$$

### Final calculation table

| Pair | Dot product | Query magnitude | Document magnitude | Cosine |
|---|---:|---:|---:|---:|
| Q and D1 | 0.78 | 0.9644 | 0.8124 | 0.9956 |
| Q and D2 | 1.08 | 0.9644 | 1.1225 | 0.9977 |
| Q and D3 | 0.34 | 0.9644 | 0.8307 | 0.4244 |

**Final ranking:**

| Document | Cosine Similarity | Rank |
|---|---:|---:|
| D2 | 0.9977 | 1 |
| D1 | 0.9956 | 2 |
| D3 | 0.4244 | 3 |

### Written answers

**1. Which document is most similar to the query?**

D2 is the most similar document because it has the highest cosine similarity of 0.9977.

**2. Is the ranking the same as BM25?**

Not exactly. BM25 without stemming gives all documents a score of zero. BM25 with stemming ranks D2 first, while D1 and D3 are tied. Cosine similarity ranks D2 first, D1 second, and D3 third.

**3. Why can BM25 and embedding retrieval disagree?**

BM25 looks for matching words. In our example, the query contains "retrieve", but D2 contains "retrieval" and "retrieves". Without stemming, BM25 does not match these words.

Embedding retrieval uses numerical vectors to represent text. It can find similar meanings without exact word matches. This is why D2 has the highest cosine similarity even though BM25 without stemming gives it a zero score.


Optional: check your Part A arithmetic here. The working above must still be written out by hand.

In [ ]:
import math
import re
import pandas as pd
from IPython.display import display

# Original documents
documents = {
    "D1": "Large language models can generate text and answer questions.",
    "D2": "Retrieval-Augmented Generation retrieves relevant documents before generating an answer.",
    "D3": "Embeddings represent text as numerical vectors."
}

query = "How does RAG retrieve information?"

k1 = 1.5
b = 0.75

# Tokenization
def tokenize(text):
    return re.findall(r"[a-z]+", text.lower())

# Given stemming rules
stems = {
    "retrieve": "retriev",
    "retrieves": "retriev",
    "retrieval": "retriev",
    "generate": "gener",
    "generating": "gener",
    "generation": "gener",
    "information": "inform",
    "does": "doe"
}

def apply_stemming(tokens):
    return [stems.get(word, word) for word in tokens]

# BM25 calculation
def calculate_bm25(use_stemming=False):

    docs = {name: tokenize(text) for name, text in documents.items()}
    q = tokenize(query)

    if use_stemming:
        docs = {name: apply_stemming(words) for name, words in docs.items()}
        q = apply_stemming(q)

    N = len(docs)
    avgdl = sum(len(words) for words in docs.values()) / N

    print("Query:", q)
    print("Average document length:", round(avgdl, 4))

    for name, words in docs.items():
        print(name, "length =", len(words), "tokens =", words)

    rows = []
    scores = {name: 0.0 for name in docs}

    for term in q:
        df = sum(term in words for words in docs.values())
        idf = math.log((N - df + 0.5) / (df + 0.5))

        row = {"Term": term}

        for name, words in docs.items():
            tf = words.count(term)
            row[name + " TF"] = tf

            denominator = tf + k1 * (
                1 - b + b * len(words) / avgdl
            )
            contribution = idf * tf * (k1 + 1) / denominator
            scores[name] += contribution

        row["DF"] = df
        row["IDF"] = round(idf, 4)
        rows.append(row)

    display(pd.DataFrame(rows))

    result = pd.DataFrame({
        "Document": list(scores.keys()),
        "BM25 Score": [round(v, 4) for v in scores.values()]
    })

    display(result.sort_values("BM25 Score", ascending=False))


print("A1(a) - WITHOUT STEMMING")
calculate_bm25(False)

print("\nA1(b) - WITH STEMMING")
calculate_bm25(True)


# A2 - Cosine similarity
print("\nA2 - COSINE SIMILARITY")

Q = [0.8, 0.2, 0.5]

vectors = {
    "D1": [0.7, 0.1, 0.4],
    "D2": [0.9, 0.3, 0.6],
    "D3": [0.1, 0.8, 0.2]
}

def magnitude(v):
    return math.sqrt(sum(x ** 2 for x in v))

results = []

for name, D in vectors.items():
    dot = sum(x * y for x, y in zip(Q, D))
    cosine = dot / (magnitude(Q) * magnitude(D))

    results.append({
        "Document": name,
        "Dot Product": round(dot, 4),
        "Query Magnitude": round(magnitude(Q), 4),
        "Document Magnitude": round(magnitude(D), 4),
        "Cosine Similarity": round(cosine, 4)
    })

display(
    pd.DataFrame(results).sort_values(
        "Cosine Similarity", ascending=False
    )
)


A1(a) - WITHOUT STEMMING
Query: ['how', 'does', 'rag', 'retrieve', 'information']
Average document length: 8.3333
D1 length = 9 tokens = ['large', 'language', 'models', 'can', 'generate', 'text', 'and', 'answer', 'questions']
D2 length = 10 tokens = ['retrieval', 'augmented', 'generation', 'retrieves', 'relevant', 'documents', 'before', 'generating', 'an', 'answer']
D3 length = 6 tokens = ['embeddings', 'represent', 'text', 'as', 'numerical', 'vectors']


,Term,D1 TF,D2 TF,D3 TF,DF,IDF
0,how,0,0,0,0,1.9459
1,does,0,0,0,0,1.9459
2,rag,0,0,0,0,1.9459
3,retrieve,0,0,0,0,1.9459
4,information,0,0,0,0,1.9459


,Document,BM25 Score
0,D1,0.0
1,D2,0.0
2,D3,0.0



A1(b) - WITH STEMMING
Query: ['how', 'doe', 'rag', 'retriev', 'inform']
Average document length: 8.3333
D1 length = 9 tokens = ['large', 'language', 'models', 'can', 'gener', 'text', 'and', 'answer', 'questions']
D2 length = 10 tokens = ['retriev', 'augmented', 'gener', 'retriev', 'relevant', 'documents', 'before', 'gener', 'an', 'answer']
D3 length = 6 tokens = ['embeddings', 'represent', 'text', 'as', 'numerical', 'vectors']


,Term,D1 TF,D2 TF,D3 TF,DF,IDF
0,how,0,0,0,0,1.9459
1,doe,0,0,0,0,1.9459
2,rag,0,0,0,0,1.9459
3,retriev,0,2,0,1,0.5108
4,inform,0,0,0,0,1.9459


,Document,BM25 Score
1,D2,0.6857
0,D1,0.0000
2,D3,0.0000



A2 - COSINE SIMILARITY


,Document,Dot Product,Query Magnitude,Document Magnitude,Cosine Similarity
1,D2,1.08,0.9644,1.1225,0.9977
0,D1,0.78,0.9644,0.8124,0.9956
2,D3,0.34,0.9644,0.8307,0.4244


---
# Part B — the same two retrievers, from libraries (1.5 pt)

Load `data/corpus.jsonl` and `data/queries.json`.

In [1]:
%cd /content

# Clone my repository
!git clone https://github.com/turkestanova2006/css4007-hw3.git

# Open repository folder
%cd /content/css4007-hw3

# Install libraries
!pip install -q -r requirements.txt


/content
Cloning into 'css4007-hw3'...
remote: Enumerating objects: 18, done.
remote: Counting objects: 100% (7/7), done.
remote: Compressing objects: 100% (7/7), done.
remote: Total 18 (delta 3), reused 0 (delta 0), pack-reused 11 (from 1)
Receiving objects: 100% (18/18), 41.79 KiB | 4.18 MiB/s, done.
Resolving deltas: 100% (5/5), done.
/content/css4007-hw3
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 17.2/17.2 MB 46.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 947.5/947.5 kB 35.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 77.5/77.5 kB 4.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.2/60.2 kB 3.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.9/4.9 MB 70.2 MB/s eta 0:00:00


In [2]:
import json
import os
import subprocess

if not os.path.exists("data/corpus.jsonl"):
    subprocess.run(
        ["git", "clone",
         "https://github.com/turkestanova2006/css4007-hw3.git",
         "/content/hw3_repo"],
        check=True
    )
    os.chdir("/content/hw3_repo")

with open("data/corpus.jsonl", "r", encoding="utf-8") as f:
    corpus = [json.loads(line) for line in f]

with open("data/queries.json", "r", encoding="utf-8") as f:
    queries = json.load(f)

print("Number of documents:", len(corpus))
print("Number of queries:", len(queries))
print("First document:", corpus[0])

Number of documents: 74
Number of queries: 2
First document: {'id': 'KZ-01', 'source': 'wikipedia', 'title': 'Kazakhstan (part 1)', 'lang': 'en', 'url': 'https://en.wikipedia.org/wiki/Kazakhstan', 'text': "Kazakhstan, officially the Republic of Kazakhstan, is a landlocked country situated primarily in Central Asia, with a portion of its territory extending into Eastern Europe. It borders Russia to the north and west, China to the east, Kyrgyzstan to the southeast, Uzbekistan to the south, and Turkmenistan to the southwest, and it has a coastline along the Caspian Sea. The capital is Astana, and the country's largest city and principal cultural and economic center is Almaty, which served as the capital until 1997."}


## B1 — BM25 with `rank_bm25`

**My tokenisation:**I use lowercase text and regex tokenisation. I do not use stemming or remove stop words. I use the same method for documents and queries.

>

In [6]:
!pip -q install rank_bm25

import re
from rank_bm25 import BM25Okapi

def tokenize(text):
    return re.findall(r"\b\w+\b", text.lower())

documents = [doc["text"] for doc in corpus]
tokenized_corpus = [tokenize(text) for text in documents]

bm25 = BM25Okapi(tokenized_corpus)

print("BM25 ready!")
print("Documents:", len(tokenized_corpus))
print("First tokens:", tokenized_corpus[0][:15])

BM25 ready!
Documents: 74
First tokens: ['kazakhstan', 'officially', 'the', 'republic', 'of', 'kazakhstan', 'is', 'a', 'landlocked', 'country', 'situated', 'primarily', 'in', 'central', 'asia']


In [7]:
# BM25: find top 3 documents for a question

def bm25_search(question, top_k=3):
    scores = bm25.get_scores(tokenize(question))
    top_indices = sorted(
        range(len(scores)),
        key=lambda i: scores[i],
        reverse=True
    )[:top_k]

    return [
        (corpus[i]["id"], round(float(scores[i]), 4))
        for i in top_indices
    ]

# Check the structure of our questions
print("Queries type:", type(queries).__name__)

if isinstance(queries, dict):
    print("Keys:", list(queries.keys()))
    for key, value in queries.items():
        print(key, ":", str(value)[:300])
else:
    print("First query:", queries[0])

Queries type: dict
Keys: ['note', 'queries']
note : Part B's eight fixed queries. Do not edit them: they are what makes every submission comparable. Add at least two of your own (Q-09, Q-10, ...) in your notebook. `relevant` lists the chunk ids that answer the query; a hit@3 is any of them in the top three.
queries : [{'id': 'Q-01', 'text': 'What is form GR-305 used for?', 'relevant': ['GO-08', 'GO-12', 'GO-23']}, {'id': 'Q-02', 'text': 'My family is poor. Is there financial help for students like me?', 'relevant': ['GO-01', 'GO-02', 'GO-03']}, {'id': 'Q-03', 'text': 'Өтінішті қашанға дейін тапсыру керек?', 'rel


In [8]:
# B1: BM25 results for all 8 questions

questions = queries["queries"]

results_bm25 = []

for q in questions:
    top3 = bm25_search(q["text"], top_k=3)

    retrieved_ids = [doc_id for doc_id, score in top3]
    relevant_ids = q["relevant"]

    hit = any(doc_id in relevant_ids for doc_id in retrieved_ids)

    results_bm25.append({
        "query": q["id"],
        "top3": retrieved_ids,
        "hit@3": hit
    })

    print("Question:", q["id"])
    print("Text:", q["text"])
    print("Top 3:", top3)
    print("Relevant:", relevant_ids)
    print("Hit@3:", hit)
    print("-" * 45)

hits = sum(result["hit@3"] for result in results_bm25)

print("BM25 Hit@3:", hits, "/", len(questions))
print("BM25 Hit@3 rate:", round(hits / len(questions), 3))

Question: Q-01
Text: What is form GR-305 used for?
Top 3: [('GO-08', 15.8255), ('GO-12', 8.1953), ('GO-06', 8.07)]
Relevant: ['GO-08', 'GO-12', 'GO-23']
Hit@3: True
---------------------------------------------
Question: Q-02
Text: My family is poor. Is there financial help for students like me?
Top 3: [('KZ-17', 8.7262), ('GO-20', 8.3105), ('KZ-25', 7.5108)]
Relevant: ['GO-01', 'GO-02', 'GO-03']
Hit@3: False
---------------------------------------------
Question: Q-03
Text: Өтінішті қашанға дейін тапсыру керек?
Top 3: [('GO-14', 10.6979), ('KZ-01', 0.0), ('KZ-02', 0.0)]
Relevant: ['GO-09', 'GO-14']
Hit@3: True
---------------------------------------------
Question: Q-04
Text: What is the minimum GPA for the grant?
Top 3: [('GO-05', 12.7798), ('GO-17', 10.5176), ('GO-19', 10.136)]
Relevant: ['GO-02']
Hit@3: False
---------------------------------------------
Question: Q-05
Text: When was Astana renamed Nur-Sultan, and why?
Top 3: [('KZ-05', 15.7315), ('KZ-06', 6.4431), ('KZ-11', 5.8659

## B2 — EmbeddingGemma (`google/embeddinggemma-300m`) with `encode_document` / `encode_query`

In [10]:

from google.colab import userdata
from sentence_transformers import SentenceTransformer
import torch

# Get Hugging Face token from Colab Secrets
hf_token = userdata.get("HF_TOKEN")

# Load EmbeddingGemma model
model = SentenceTransformer(
    "google/embeddinggemma-300m",
    token=hf_token,
    device="cuda" if torch.cuda.is_available() else "cpu"
)

print("EmbeddingGemma loaded successfully!")
print("Device:", model.device)

modules.json:   0%|          | 0.00/573 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/997 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/18.7k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/58.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/1.49k [00:00<?, ?B/s]

...

2_Dense/model.safetensors: downloading bytes:           |  0.00B            

config.json:   0%|          | 0.00/134 [00:00<?, ?B/s]

3_Dense/model.safetensors: reconstructing file:   0%|          |  0.00B / 9.44MB            

3_Dense/model.safetensors: downloading bytes:           |  0.00B            

EmbeddingGemma loaded successfully!
Device: cpu


In [11]:
import numpy as np

# Create embeddings for 74 documents
doc_embeddings = model.encode_document(
    documents,
    batch_size=4,
    show_progress_bar=True,
    normalize_embeddings=True
)

# Create embeddings for 8 questions
question_texts = [q["text"] for q in questions]

query_embeddings = model.encode_query(
    question_texts,
    batch_size=4,
    show_progress_bar=True,
    normalize_embeddings=True
)

print("Document embeddings:", doc_embeddings.shape)
print("Query embeddings:", query_embeddings.shape)

Batches:   0%|          | 0/19 [00:00<?, ?it/s]

Batches:   0%|          | 0/2 [00:00<?, ?it/s]

Document embeddings: (74, 768)
Query embeddings: (8, 768)




### My own queries

| Id | Query | relevant | Expected winner, and why |
|---|---|---|---|
| Q-09 | Which form is used for the original need-based grant application? | GO-06, GO-14 | BM25. The question uses specific words such as "form" and "application". |
| Q-10 | Can a student who takes all classes remotely qualify for this financial support? | GO-20 | Embeddings. The question uses "remotely", but the document uses "distance-learning". The meaning is similar. |

In [16]:
for doc in corpus:
    if doc["id"] in ["GO-06", "GO-14", "GO-20"]:
        print("ID:", doc["id"])
        print("TEXT:", doc["text"])
        print("-" * 50)

ID: GO-06
TEXT: Form GR-104 is the application form for the need-based study grant. It is the only way to apply; an email or a letter is not an application. GR-104 is filled in on the student portal and asks for the student's id, programme, the bank account for payment, and a declaration that the information is true. The transcript and the id card copy are uploaded as attachments to GR-104.
--------------------------------------------------
ID: GO-14
TEXT: 2026 жылғы грантқа өтініштер 1 қыркүйектен бастап 15 қазанға дейін студенттік портал арқылы қабылданады. Өтініш беру үшін GR-104 нысанын толтырып, транскрипт пен жеке куәліктің көшірмесін тіркеу керек. Мерзімі өткен өтініштер қаралмайды. Сұрақтарыңыз болса, Грант бөліміне хабарласыңыз.
--------------------------------------------------
ID: GO-20
TEXT: The need-based grant is for full-time undergraduate students only. Part-time students, distance-learning students and incoming exchange students cannot apply. Master's and doctoral stud

In [17]:
own_queries = [
    {
        "id": "Q-09",
        "text": "Which form is used for the original need-based grant application?",
        "relevant": ["GO-06", "GO-14"]
    },
    {
        "id": "Q-10",
        "text": "Can a student who takes all classes remotely qualify for this financial support?",
        "relevant": ["GO-20"]
    }
]

all_questions = questions + own_queries

print("Total questions:", len(all_questions))

for q in own_queries:
    print(q["id"], ":", q["text"])

Total questions: 10
Q-09 : Which form is used for the original need-based grant application?
Q-10 : Can a student who takes all classes remotely qualify for this financial support?


## Results for all ten or more queries: top 3 from each retriever

In [19]:
import numpy as np
import pandas as pd

# Create embeddings for all 10 questions
all_query_embeddings = model.encode_query(
    [q["text"] for q in all_questions],
    batch_size=4,
    normalize_embeddings=True,
    show_progress_bar=True
)

results = []

for i, q in enumerate(all_questions):

    # BM25 Top 3
    bm25_top = bm25_search(q["text"], top_k=3)
    bm25_ids = [doc_id for doc_id, score in bm25_top]

    # EmbeddingGemma Top 3
    scores = all_query_embeddings[i] @ doc_embeddings.T
    top_indices = np.argsort(scores)[::-1][:3]
    emb_ids = [corpus[j]["id"] for j in top_indices]

    # Check Hit@3
    bm25_hit = any(x in q["relevant"] for x in bm25_ids)
    emb_hit = any(x in q["relevant"] for x in emb_ids)

    results.append({
        "Query": q["id"],
        "BM25 Top 3": ", ".join(bm25_ids),
        "BM25 Hit@3": bm25_hit,
        "Embedding Top 3": ", ".join(emb_ids),
        "Embedding Hit@3": emb_hit
    })

df = pd.DataFrame(results)

display(df)

print("BM25 Hit@3:", df["BM25 Hit@3"].mean())
print("Embedding Hit@3:", df["Embedding Hit@3"].mean())

Batches:   0%|          | 0/3 [00:00<?, ?it/s]

,Query,BM25 Top 3,BM25 Hit@3,Embedding Top 3,Embedding Hit@3
0,Q-01,"GO-08, GO-12, GO-06",True,"GO-08, GO-12, GO-23",True
1,Q-02,"KZ-17, GO-20, KZ-25",False,"GO-01, GO-10, GO-20",True
2,Q-03,"GO-14, KZ-01, KZ-02",True,"GO-14, GO-09, GO-11",True
3,Q-04,"GO-05, GO-17, GO-19",False,"GO-02, GO-05, GO-17",True
4,Q-05,"KZ-05, KZ-06, KZ-11",True,"KZ-05, KZ-06, KZ-03",True
5,Q-06,"KZ-37, KZ-38, KZ-03",True,"KZ-37, KZ-38, KZ-10",True
6,Q-07,"GO-21, GO-16, GO-01",False,"GO-15, GO-18, GO-01",True
7,Q-08,"GO-17, GO-05, GO-04",True,"GO-17, GO-18, GO-15",True
8,Q-09,"GO-06, GO-16, GO-08",True,"GO-06, GO-20, GO-01",True
9,Q-10,"GO-16, GO-02, GO-18",False,"GO-20, GO-02, GO-01",True


BM25 Hit@3: 0.6
Embedding Hit@3: 1.0


## B3 — comparison

### B3 — comparison

| Query | BM25 top 3 (ids) | BM25 hit@3 | Embedding top 3 (ids) | Embedding hit@3 |
|---|---|---|---|---|
| Q-01 | GO-08, GO-12, GO-06 | True | GO-08, GO-12, GO-23 | True |
| Q-02 | KZ-17, GO-20, KZ-25 | False | GO-01, GO-10, GO-20 | True |
| Q-03 | GO-14, KZ-01, KZ-02 | True | GO-14, GO-09, GO-11 | True |
| Q-04 | GO-05, GO-17, GO-19 | False | GO-02, GO-05, GO-17 | True |
| Q-05 | KZ-05, KZ-06, KZ-11 | True | KZ-05, KZ-06, KZ-03 | True |
| Q-06 | KZ-37, KZ-38, KZ-03 | True | KZ-37, KZ-38, KZ-10 | True |
| Q-07 | GO-21, GO-16, GO-01 | False | GO-15, GO-18, GO-01 | True |
| Q-08 | GO-17, GO-05, GO-04 | True | GO-17, GO-18, GO-15 | True |
| Q-09 | GO-06, GO-16, GO-08 | True | GO-06, GO-20, GO-01 | True |
| Q-10 | GO-16, GO-02, GO-18 | False | GO-20, GO-02, GO-01 | True |
| **hit@3 total** | | **6/10** | | **10/10** |


**Both retrievers agree: which query, and why was it easy?**

Q-05. Both methods found KZ-05 as the first result. The question has clear words about Astana and Nur-Sultan, so it was easy to find the correct document.

**BM25 did better: which query, and what exact string did it match that the embedding model blurred?**

There was no query where BM25 had a higher Hit@3 than EmbeddingGemma. For Q-09, both methods found GO-06, which contains information about the grant application form. BM25 can match exact words like "form" and "application".

**Embeddings did better: which query, and what could BM25 not match?**

Q-10. EmbeddingGemma found GO-20, but BM25 did not. The question uses "remotely", while the document uses "distance-learning". EmbeddingGemma understood the similar meaning.

>

---
# Part C — connect retrieval to the model (1.5 pt)

Model: `gpt-5.6-luna` for every call. Top 3 unless stated.

In [20]:
%pip -q install openai tiktoken

In [23]:
import tiktoken

# Choose a question
question = "What is Form GR-305 used for?"

# Find top 3 documents using BM25
top3 = bm25_search(question, top_k=3)

# Build context
context_parts = []

for doc_id, score in top3:
    doc = next(d for d in corpus if d["id"] == doc_id)
    context_parts.append(
        f"[{doc_id}]\n{doc['text']}"
    )

context = "\n\n".join(context_parts)

# System instructions
system_prompt = (
    "You are a helpful assistant. "
    "Answer the question using only the provided context. "
    "If the answer is not in the context, say you do not know. "
    "Cite the document IDs in your answer."
)

# User prompt
user_prompt = (
    f"Context:\n{context}\n\n"
    f"Question: {question}\n"
    "Answer:"
)

# Print complete prompt before sending
print("=== SYSTEM INSTRUCTIONS ===")
print(system_prompt)

print("\n=== USER PROMPT ===")
print(user_prompt)

# Count tokens
enc = tiktoken.get_encoding("o200k_base")

system_tokens = len(enc.encode(system_prompt))
question_tokens = len(enc.encode(question))
context_tokens = len(enc.encode(context))

print("\n=== TOKEN COUNT ===")
print("System instructions:", system_tokens)
print("Question:", question_tokens)
print("Retrieved context:", context_tokens)
print("Total content tokens:",
      system_tokens + question_tokens + context_tokens)

print("\nNumber of retrieved chunks:", len(top3))
print("Context characters:", len(context))

=== SYSTEM INSTRUCTIONS ===
You are a helpful assistant. Answer the question using only the provided context. If the answer is not in the context, say you do not know. Cite the document IDs in your answer.

=== USER PROMPT ===
Context:
[GO-08]
Form GR-305 is used to appeal a decision of the Grant Office. It is the appeal form and nothing else: it cannot be used to apply for the first time or to report a change in circumstances. On GR-305 the student states which part of the decision they believe is wrong and which rule in this handbook supports their case.

[GO-12]
A student who disagrees with a decision may appeal within 14 calendar days of the date on the decision letter, using form GR-305. The appeal is read by a panel of three staff members who did not take part in the original decision, and the panel answers within 20 working days. An appeal reviews the decision on the evidence that was on file; new income information cannot be added through an appeal and needs a new application.


In [29]:
from google.colab import userdata

import os

api_key = userdata.get("OPENAI_API_KEY")

if api_key:
    os.environ["OPENAI_API_KEY"] = api_key
    print("API key found!")
else:
    print("API key not found!")

API key found!


## C1 — the complete prompt, printed before it is sent

In [30]:
from openai import OpenAI

client = OpenAI()

# Model required by the assignment
model_name = "gpt-5.6-luna"

try:
    response = client.chat.completions.create(
        model=model_name,
        messages=[
            {"role": "system", "content": system_prompt},
            {"role": "user", "content": user_prompt}
        ]
    )

    print("=== MODEL ANSWER ===")
    print(response.choices[0].message.content)

    print("\n=== TOKEN USAGE ===")
    print("Prompt tokens:", response.usage.prompt_tokens)
    print("Completion tokens:", response.usage.completion_tokens)
    print("Total tokens:", response.usage.total_tokens)

except Exception as e:
    print("API error:", str(e))

=== MODEL ANSWER ===
Form GR-305 is used only to appeal a Grant Office decision. It cannot be used for a first-time application or to report changed circumstances. [GO-08]

=== TOKEN USAGE ===
Prompt tokens: 317
Completion tokens: 37
Total tokens: 354


| Part of the input | Tokens (tiktoken `o200k_base`) |
|---|---:|
| System instructions | 38 |
| Question | 9 |
| Retrieved context | 254 |
| **Total input** | **301** |
| **`usage.prompt_tokens` from the response** | **317** |

- Number of retrieved chunks: 3
- Tokens per chunk: Not measured separately yet.
- Total context size (tokens / characters): 254 tokens / 1150 characters

**Why do the tiktoken total and `usage.prompt_tokens` differ?**

> My local count is 301 tokens, while the API reports 317 tokens. The difference is 16 tokens because my calculation does not include all message formatting and prompt text.

**What exactly does retrieval add to the model's input?**

> Retrieval adds three relevant document chunks to the prompt: GO-08, GO-12, and GO-06. These documents give the model information about Form GR-305 and help it answer the question using the provided context.

In [31]:
for doc_id, score in top3:
    doc = next(d for d in corpus if d["id"] == doc_id)
    chunk_text = f"[{doc_id}]\n{doc['text']}"
    print(doc_id, len(enc.encode(chunk_text)), "tokens")

GO-08 73 tokens
GO-12 93 tokens
GO-06 88 tokens


## C2 — three systems, six questions

In [32]:
# Check the structure of the C2 questions

print("Type:", type(queries))

if isinstance(queries, dict):
    print("Keys:", list(queries.keys()))
    for key, value in queries.items():
        print(key, ":", str(value)[:800])
else:
    print("First items:", queries[:2])

Type: <class 'dict'>
Keys: ['note', 'queries']
note : Part B's eight fixed queries. Do not edit them: they are what makes every submission comparable. Add at least two of your own (Q-09, Q-10, ...) in your notebook. `relevant` lists the chunk ids that answer the query; a hit@3 is any of them in the top three.
queries : [{'id': 'Q-01', 'text': 'What is form GR-305 used for?', 'relevant': ['GO-08', 'GO-12', 'GO-23']}, {'id': 'Q-02', 'text': 'My family is poor. Is there financial help for students like me?', 'relevant': ['GO-01', 'GO-02', 'GO-03']}, {'id': 'Q-03', 'text': 'Өтінішті қашанға дейін тапсыру керек?', 'relevant': ['GO-09', 'GO-14']}, {'id': 'Q-04', 'text': 'What is the minimum GPA for the grant?', 'relevant': ['GO-02']}, {'id': 'Q-05', 'text': 'When was Astana renamed Nur-Sultan, and why?', 'relevant': ['KZ-05']}, {'id': 'Q-06', 'text': 'Where did the Soviet Union test its nuclear weapons?', 'relevant': ['KZ-37', 'KZ-38']}, {'id': 'Q-07', 'text': "Can the grant be sent to my mo

In [33]:
import os

for root, dirs, files in os.walk("/content/hw3_repo"):
    for file in files:
        if file.endswith((".json", ".jsonl")):
            print(os.path.join(root, file))

In [34]:
# Check variables for C2

needed = [
    "qa",
    "client",
    "MODEL_NAME",
    "system_message",
    "bm25_search",
    "embedding_search"
]

for name in needed:
    print(name, "OK" if name in globals() else "MISSING")

qa OK
client OK
MODEL_NAME MISSING
system_message MISSING
bm25_search OK
embedding_search MISSING


In [35]:
# Model for Part C
MODEL_NAME = "gpt-5.6-luna"

# System instructions
system_message = (
    "Answer the question using only the provided context. "
    "If the answer is not in the context, say you do not know. "
    "Cite the chunk IDs used."
)

# Embedding search
import numpy as np

def embedding_search(question, top_k=3):
    query_vector = model.encode_query(
        [question],
        normalize_embeddings=True
    )[0]

    scores = doc_embeddings @ query_vector

    top_indices = np.argsort(scores)[::-1][:top_k]

    results = []

    for idx in top_indices:
        doc = corpus[int(idx)]

        results.append({
            "id": doc["id"],
            "text": doc["text"],
            "score": float(scores[idx])
        })

    return results

print("MODEL_NAME ready!")
print("system_message ready!")
print("embedding_search ready!")

MODEL_NAME ready!
system_message ready!
embedding_search ready!


In [37]:
# C2 — Three systems, six questions

def build_rag_messages(question, chunks):
    blocks = []

    for i, doc in enumerate(chunks, 1):
        blocks.append(
            f"[{i}] Chunk ID: {doc['id']}\n{doc['text']}"
        )

    context = "Context:\n\n" + "\n\n".join(blocks)

    return [
        {"role": "system", "content": system_message},
        {"role": "user", "content": context + "\n\nQuestion: " + question}
    ]


c2_results = {}

for q in qa:
    qid = q["id"]
    question = q["question"]

    print("\nQUESTION:", qid, question)
    print("REFERENCE:", q["reference"])

    # Convert BM25 tuple results (id, score) to document objects from corpus
    bm25_hits = bm25_search(question, top_k=3)
    bm25_chunks = []
    for doc_id, score in bm25_hits:
        doc_obj = next(d for d in corpus if d["id"] == doc_id)
        bm25_chunks.append({
            "id": doc_obj["id"],
            "text": doc_obj["text"]
        })

    systems = {
        "LLM only": [
            {"role": "user", "content": question}
        ],
        "BM25 RAG": build_rag_messages(
            question, bm25_chunks
        ),
        "Embedding RAG": build_rag_messages(
            question, embedding_search(question, top_k=3)
        )
    }

    c2_results[qid] = {}

    for name, messages in systems.items():
        response = client.chat.completions.create(
            model=MODEL_NAME,
            messages=messages
        )

        answer = response.choices[0].message.content
        c2_results[qid][name] = answer

        print("\nSYSTEM:", name)
        print("ANSWER:", answer)

print("\nAll 18 answers completed!")


QUESTION: C-01 How much does a student in income band 2 receive, and when is it paid?
REFERENCE: 150,000 tenge for the academic year, in two equal instalments: October and February.

SYSTEM: LLM only
ANSWER: Which student support scheme or country do you mean? “Income band 2” is used by several student-finance systems, and the amount and payment schedule vary. Please share the name of the scheme or a link/document.

SYSTEM: BM25 RAG
ANSWER: A student in income band 2 receives **150,000 tenge for the academic year**, paid in **two equal instalments**: the first in **October** and the second in **February**. [GO-04]

SYSTEM: Embedding RAG
ANSWER: A student in income band 2 receives **150,000 tenge for the academic year**, paid in **two equal instalments: 75,000 tenge in October and 75,000 tenge in February**. [GO-04]

QUESTION: C-02 My grant application was refused. How long do I have to appeal, and which form do I use?
REFERENCE: 14 calendar days from the date on the decision letter, u

In [38]:
import pandas as pd
from IPython.display import display

rows = []

for q in qa:
    qid = q["id"]

    rows.append({
        "Question": qid,
        "Reference": q["reference"],
        "LLM only": c2_results[qid]["LLM only"],
        "BM25 RAG": c2_results[qid]["BM25 RAG"],
        "Embedding RAG": c2_results[qid]["Embedding RAG"]
    })

c2_table = pd.DataFrame(rows)

pd.set_option("display.max_colwidth", 200)

display(c2_table)

,Question,Reference,LLM only,BM25 RAG,Embedding RAG
0,C-01,"150,000 tenge for the academic year, in two equal instalments: October and February.","Which student support scheme or country do you mean? “Income band 2” is used by several student-finance systems, and the amount and payment schedule vary. Please share the name of the scheme or a ...","A student in income band 2 receives **150,000 tenge for the academic year**, paid in **two equal instalments**: the first in **October** and the second in **February**. [GO-04]","A student in income band 2 receives **150,000 tenge for the academic year**, paid in **two equal instalments: 75,000 tenge in October and 75,000 tenge in February**. [GO-04]"
1,C-02,"14 calendar days from the date on the decision letter, using form GR-305.",Appeal deadlines and forms depend on the grant program and the country or agency that issued the refusal. The refusal letter should state:\n\n- the deadline for requesting a review or appeal;\n- w...,You have **14 calendar days** from the date on the decision letter to appeal. Use **form GR-305**. [GO-12],"You have **14 calendar days from the date on the decision letter** to appeal. Use **form GR-305**, the Grant Office appeal form. [GO-12] [GO-08]"
2,C-03,A cumulative GPA of at least 2.67 on the 4.0 scale. (The 3.0 in GO-05 is the archived 2025 rule.),"The minimum GPA varies by the specific need-based grant and institution. Please share the grant’s name or the school/program, and I can identify the exact requirement.",The minimum GPA is **2.67 on a 4.0 scale**. ([GO-02]),The minimum GPA is **2.67 on a 4.0 scale**. [GO-02]
3,C-04,A presidential decree of October 2017 ordered the change for 2025; in January 2021 the target was moved to 2031.,"Kazakhstan formally decided on the change on **26 October 2017**, when President Nursultan Nazarbayev signed a decree beginning the transition from Cyrillic to a Latin-based Kazakh alphabet. The o...","Kazakhstan decided to switch from Cyrillic to Latin in **October 2017**. The transition was initially due to finish by **2025**, but the target was later postponed to **2031**. [KZ-43]","Kazakhstan decided to switch from Cyrillic to Latin in **October 2017**. The transition is now due to finish in **2031**. (KZ-14, KZ-43)"
4,C-05,"In 1963, as the Almaty Institute of National Economy; it took its current name in 2016.",Narxoz University was founded in **1963** as the **Alma-Ata Institute of National Economy**.,Narxoz University was founded in **1963** as the **Almaty Institute of National Economy**. (KZ-44),Narxoz University was founded in **1963** as the **Almaty Institute of National Economy**. (KZ-44)
5,C-06,The corpus does not say. GO-20 says master's students have a separate scheme that the handbook does not describe.,"There isn’t one universal amount—need-based master’s grants depend mainly on your **country, university, program, enrollment, and financial circumstances**.\n\n**If you’re in the U.S.:**\n- Most f...","I do not know. The context says that master's students are supported by a separate scheme, but it does not state that scheme’s grant amount. [GO-20]",You cannot apply for this need-based grant because it is only for full-time undergraduate students. Master’s students are covered by a separate scheme and should ask their department for its rules...


In [39]:
# C2 — Check all answers against references

for q in qa:
    qid = q["id"]

    print("\n" + "=" * 60)
    print("QUESTION:", qid)
    print("REFERENCE:", q["reference"])

    for system in ["LLM only", "BM25 RAG", "Embedding RAG"]:
        print("\n", system)
        print(c2_results[qid][system])

    print("=" * 60)


QUESTION: C-01
REFERENCE: 150,000 tenge for the academic year, in two equal instalments: October and February.

 LLM only
Which student support scheme or country do you mean? “Income band 2” is used by several student-finance systems, and the amount and payment schedule vary. Please share the name of the scheme or a link/document.

 BM25 RAG
A student in income band 2 receives **150,000 tenge for the academic year**, paid in **two equal instalments**: the first in **October** and the second in **February**. [GO-04]

 Embedding RAG
A student in income band 2 receives **150,000 tenge for the academic year**, paid in **two equal instalments: 75,000 tenge in October and 75,000 tenge in February**. [GO-04]

QUESTION: C-02
REFERENCE: 14 calendar days from the date on the decision letter, using form GR-305.

 LLM only
Appeal deadlines and forms depend on the grant program and the country or agency that issued the refusal. The refusal letter should state:

- the deadline for requesting a revi

In [40]:
for q in qa[3:6]:
    qid = q["id"]

    print("\n" + "=" * 50)
    print("QUESTION:", qid, q["question"])
    print("REFERENCE:", q["reference"])

    for system in ["LLM only", "BM25 RAG", "Embedding RAG"]:
        print("\nSYSTEM:", system)
        print("ANSWER:", c2_results[qid][system])


QUESTION: C-04 When did Kazakhstan decide to switch the Kazakh alphabet from Cyrillic to Latin, and when is the switch now due to finish?
REFERENCE: A presidential decree of October 2017 ordered the change for 2025; in January 2021 the target was moved to 2031.

SYSTEM: LLM only
ANSWER: Kazakhstan formally decided on the change on **26 October 2017**, when President Nursultan Nazarbayev signed a decree beginning the transition from Cyrillic to a Latin-based Kazakh alphabet. The original target was **2025**.

That deadline was later extended: the phased transition is now generally scheduled to be completed by **2031**.

SYSTEM: BM25 RAG
ANSWER: Kazakhstan decided to switch from Cyrillic to Latin in **October 2017**. The transition was initially due to finish by **2025**, but the target was later postponed to **2031**. [KZ-43]

SYSTEM: Embedding RAG
ANSWER: Kazakhstan decided to switch from Cyrillic to Latin in **October 2017**. The transition is now due to finish in **2031**. (KZ-14, K

| Question | LLM only | BM25 RAG | Embedding RAG |
|---|---|---|---|
| C-01 | ✗ No exact amount | ✓ 150,000 KZT | ✓ 150,000 KZT |
| C-02 | ✗ No exact deadline | ✓ 14 days, GR-305 | ✓ 14 days, GR-305 |
| C-03 | ✗ No exact GPA | ✓ GPA 2.67 | ✓ GPA 2.67 |
| C-04 | To check | To check | To check |
| C-05 | To check | To check | To check |
| C-06 | To check | ✓ Amount not provided | To check |
| **Agrees with reference** | **Pending /6** | **Pending /6** | **Pending /6** |

## C3 — question 1 of 3: (id)

**1. What BM25 retrieved (top 3, ids and scores):**
- GO-04: 20.2192
- GO-02: 18.7422
- GO-01: 17.243

**2. What EmbeddingGemma retrieved (top 3, ids and scores):**
- GO-04
- GO-03
- GO-15

The embedding scores were not printed in this output.

**3. What was sent to the model (the context blocks):**
BM25 RAG used GO-04, GO-02 and GO-01.
Embedding RAG used GO-04, GO-03 and GO-15.

**4. How retrieval changed the answer:**
Without retrieval, the LLM asked for more information. Both RAG systems gave the correct amount: 150,000 tenge per academic year, paid in two equal parts in October and February. Retrieval made the answers more specific and accurate.

**5. What was missed, and was it a retrieval problem or a generation problem? Quote the evidence:**
The LLM-only answer missed the amount and payment dates because it did not have the grant documents. Both RAG systems found useful information. The reference says: "150,000 tenge for the academic year, in two equal instalments: October and February."

In [41]:
# C3 - Question 1: C-01

q = qa[0]
question = q["question"]

print("QUESTION:", question)
print("REFERENCE:", q["reference"])

print("\n=== BM25 TOP 3 ===")
bm = bm25_search(question, top_k=3)
for doc in bm:
    print(doc)

print("\n=== EMBEDDING TOP 3 ===")
emb = embedding_search(question, top_k=3)
for doc in emb:
    print(doc)

print("\n=== MODEL ANSWERS ===")
for name, answer in c2_results["C-01"].items():
    print("\n", name, ":", answer)

QUESTION: How much does a student in income band 2 receive, and when is it paid?
REFERENCE: 150,000 tenge for the academic year, in two equal instalments: October and February.

=== BM25 TOP 3 ===
('GO-04', 20.2192)
('GO-02', 18.7422)
('GO-01', 17.243)

=== EMBEDDING TOP 3 ===
{'id': 'GO-04', 'text': 'The grant amount depends only on the income band, not on the GPA. A student in band 1 receives 250,000 tenge for the academic year. A student in band 2 receives 150,000 tenge for the academic year. The amount is paid in two equal instalments, the first in October and the second in February. There is no partial grant: a student either receives the full amount for their band or nothing.', 'score': 0.584521472454071}
{'id': 'GO-03', 'text': 'The income band is set from the monthly household income per person, declared on form GR-210. Band 1: below 85,000 tenge per person per month. Band 2: from 85,000 up to 150,000 tenge per person per month. Band 3: above 150,000 tenge per person per month.

In [42]:
question = qa[0]["question"]

query_vector = model.encode_query(
    [question],
    normalize_embeddings=True
)

scores = doc_embeddings @ query_vector[0]

top_indices = scores.argsort()[::-1][:3]

for i in top_indices:
    print(corpus[i]["id"], round(float(scores[i]), 4))

GO-04 0.5845
GO-03 0.4842
GO-15 0.4506



### C3 — Question 2 of 3: C-02

**1. What BM25 retrieved (top 3, ids and scores):**

- GO-05: 17.2414
- GO-08: 14.5287
- GO-12: 13.5735

**2. What EmbeddingGemma retrieved (top 3, ids and scores):**

- GO-08
- GO-12
- GO-11

Scores will be added after running the code.

**3. What was sent to the model (the context blocks):**

BM25 RAG used GO-05, GO-08 and GO-12.

Embedding RAG used GO-08, GO-12 and GO-11.

**4. How retrieval changed the answer:**

The LLM only gave a general answer and asked for more information. Both RAG systems gave the correct answer: 14 calendar days from the decision letter and form GR-305. Retrieval made the answer more accurate and specific.

**5. What was missed, and was it a retrieval problem or a generation problem? Quote the evidence:**

The LLM-only answer missed the exact deadline and form because it had no documents. Both RAG systems retrieved useful information and answered correctly.

Evidence from GO-12: "A student who disagrees with a decision may appeal within 14 calendar days".

Evidence from GO-08: "Form GR-305 is used to appeal a decision of the Grant Office".

In [43]:
q = qa[1]

print("QUESTION:", q["question"])
print("REFERENCE:", q["reference"])

print("\n=== BM25 TOP 3 ===")
for item in bm25_search(q["question"], top_k=3):
    print(item)

print("\n=== EMBEDDING TOP 3 ===")
for item in embedding_search(q["question"], top_k=3):
    print(item)

print("\n=== MODEL ANSWERS ===")
for name, answer in c2_results[q["id"]].items():
    print(name, ":", answer)

QUESTION: My grant application was refused. How long do I have to appeal, and which form do I use?
REFERENCE: 14 calendar days from the date on the decision letter, using form GR-305.

=== BM25 TOP 3 ===
('GO-05', 17.2414)
('GO-08', 14.5287)
('GO-12', 13.5735)

=== EMBEDDING TOP 3 ===
{'id': 'GO-08', 'text': 'Form GR-305 is used to appeal a decision of the Grant Office. It is the appeal form and nothing else: it cannot be used to apply for the first time or to report a change in circumstances. On GR-305 the student states which part of the decision they believe is wrong and which rule in this handbook supports their case.', 'score': 0.6028919219970703}
{'id': 'GO-12', 'text': 'A student who disagrees with a decision may appeal within 14 calendar days of the date on the decision letter, using form GR-305. The appeal is read by a panel of three staff members who did not take part in the original decision, and the panel answers within 20 working days. An appeal reviews the decision on the

In [44]:
q = qa[1]["question"]

query_vector = model.encode_query(
    [q],
    normalize_embeddings=True
)

scores = doc_embeddings @ query_vector[0]
top_indices = scores.argsort()[::-1][:3]

for i in top_indices:
    print(corpus[i]["id"], round(float(scores[i]), 4))

GO-08 0.6029
GO-12 0.5591
GO-11 0.5343


### C3 — Question 3 of 3: C-03

**1. What BM25 retrieved (top 3, ids and scores):**

- GO-16: 15.7985
- GO-02: 14.0587
- GO-06: 13.8516

**2. What EmbeddingGemma retrieved (top 3, ids and scores):**

- GO-02: 0.6286
- GO-20: 0.5691
- GO-05: 0.5451

**3. What was sent to the model (the context blocks):**

BM25 RAG used GO-16, GO-02 and GO-06.

Embedding RAG used GO-02, GO-20 and GO-05.

GO-02 contains the correct GPA requirement for 2026. GO-05 contains the old 2025 rule.

**4. How retrieval changed the answer:**

The LLM only gave a general answer without the exact GPA. Both RAG systems correctly answered 2.67 on a 4.0 scale. Retrieval helped the model give a specific and correct answer.

**5. What was missed, and was it a retrieval problem or a generation problem? Quote the evidence:**

The LLM-only system missed the exact GPA because it had no context. Both RAG systems found GO-02 and answered correctly.

The reference says: "A cumulative GPA of at least 2.67 on the 4.0 scale."

Embedding retrieval also returned GO-05 with the old 2025 rule, but the model still used the correct 2026 information. There was no major generation error in the RAG answers.

In [45]:
q = qa[2]

print("QUESTION:", q["question"])
print("REFERENCE:", q["reference"])

print("\n=== BM25 TOP 3 ===")
for item in bm25_search(q["question"], top_k=3):
    print(item)

print("\n=== EMBEDDING TOP 3 ===")
for item in embedding_search(q["question"], top_k=3):
    print(item)

print("\n=== EMBEDDING SCORES ===")
query_vector = model.encode_query(
    [q["question"]],
    normalize_embeddings=True
)

scores = doc_embeddings @ query_vector[0]

for i in scores.argsort()[::-1][:3]:
    print(corpus[i]["id"], round(float(scores[i]), 4))

print("\n=== MODEL ANSWERS ===")
for name, answer in c2_results[q["id"]].items():
    print(name, ":", answer)

QUESTION: What is the minimum GPA needed to qualify for the need-based grant?
REFERENCE: A cumulative GPA of at least 2.67 on the 4.0 scale. (The 3.0 in GO-05 is the archived 2025 rule.)

=== BM25 TOP 3 ===
('GO-16', 15.7985)
('GO-02', 14.0587)
('GO-06', 13.8516)

=== EMBEDDING TOP 3 ===
{'id': 'GO-02', 'text': 'An applicant qualifies for the 2026 grant when all three conditions hold at the time of the decision: their cumulative GPA is at least 2.67 on the 4.0 scale; their household is in income band 1 or income band 2; and both a transcript and a copy of the national id card are on file with the Grant Office. Meeting two conditions out of three is not enough. An applicant who is not on the Grant Office record does not qualify, whatever their message says.', 'score': 0.6285815238952637}
{'id': 'GO-20', 'text': "The need-based grant is for full-time undergraduate students only. Part-time students, distance-learning students and incoming exchange students cannot apply. Master's and docto

### Final Questions

**1. What is the main difference between BM25 and embedding retrieval?**

BM25 searches for matching words. Embeddings search for similar meanings, even when the words are different.

**2. When can BM25 perform better than embeddings?**

BM25 can work better when a question has exact words, numbers, form names or document IDs.

**3. When can embeddings perform better than BM25?**

Embeddings can work better when the question and document use different words with the same meaning.

**4. Why is cosine similarity used with embeddings?**

Cosine similarity measures how similar two vectors are by comparing their directions. A higher similarity means the texts are more related.

**5. What information is actually sent to an LLM in a RAG system?**

The LLM receives system instructions, the user's question and retrieved text chunks. It does not receive the whole database.

**6. Does the LLM search the original documents itself?**

No. The retriever searches the documents first. Then the LLM uses the retrieved text to generate an answer.

**7. What happens if retrieval returns an irrelevant chunk?**

The LLM may give a wrong answer or miss important information because the retrieved context is not useful.

In [47]:
# Final Question 8: Compare Top-K

q = qa[1]
question = q["question"]

print("Question:", question)
print("Reference:", q["reference"])

for k in [1, 3, 8]:
    print("\n" + "=" * 40)
    print("TOP-K =", k)

    bm25_docs = bm25_search(question, top_k=k)
    emb_docs = embedding_search(question, top_k=k)

    print("\nBM25:")
    for doc in bm25_docs:
        print(doc[0])

    print("\nEmbedding:")
    for doc in emb_docs:
        print(doc["id"])

Question: My grant application was refused. How long do I have to appeal, and which form do I use?
Reference: 14 calendar days from the date on the decision letter, using form GR-305.

TOP-K = 1

BM25:
GO-05

Embedding:
GO-08

TOP-K = 3

BM25:
GO-05
GO-08
GO-12

Embedding:
GO-08
GO-12
GO-11

TOP-K = 8

BM25:
GO-05
GO-08
GO-12
GO-06
GO-11
GO-13
KZ-14
GO-07

Embedding:
GO-08
GO-12
GO-11
GO-23
GO-06
GO-09
GO-17
GO-21


In [49]:
# Final Question 8: Test RAG answers for K = 1, 3, 8

q = qa[1]
question = q["question"]

for k in [1, 3, 8]:
    print("\n" + "=" * 50)
    print("TOP-K =", k)

    # BM25 returns tuples, convert them to document objects with 'id' and 'text'
    bm25_hits = bm25_search(question, top_k=k)
    bm25_chunks = []
    for doc_id, score in bm25_hits:
        doc_obj = next(d for d in corpus if d["id"] == doc_id)
        bm25_chunks.append({
            "id": doc_obj["id"],
            "text": doc_obj["text"]
        })

    # Embedding RAG already returns dictionaries with 'id' and 'text'
    embedding_chunks = embedding_search(question, top_k=k)

    for name, chunks in [
        ("BM25 RAG", bm25_chunks),
        ("Embedding RAG", embedding_chunks)
    ]:
        messages = build_rag_messages(question, chunks)

        response = client.chat.completions.create(
            model=MODEL_NAME,
            messages=messages
        )

        answer = response.choices[0].message.content

        print("\nSYSTEM:", name)
        print("ANSWER:", answer)
        print("Chunks:", [d["id"] for d in chunks])


TOP-K = 1

SYSTEM: BM25 RAG
ANSWER: I do not know. The provided context does not specify the appeal deadline or the form to use. [GO-05]
Chunks: ['GO-05']

SYSTEM: Embedding RAG
ANSWER: The appeal deadline is not specified in the provided context. Use **Form GR-305** to appeal the refusal. [GO-08]
Chunks: ['GO-08']

TOP-K = 3

SYSTEM: BM25 RAG
ANSWER: You have **14 calendar days from the date on the decision letter** to appeal. Use **form GR-305**. [GO-12]
Chunks: ['GO-05', 'GO-08', 'GO-12']

SYSTEM: Embedding RAG
ANSWER: You have **14 calendar days from the date on the decision letter** to appeal. Use **form GR-305**. [GO-12]
Chunks: ['GO-08', 'GO-12', 'GO-11']

TOP-K = 8

SYSTEM: BM25 RAG
ANSWER: You have **14 calendar days from the date on the decision letter** to appeal. Use **form GR-305**, the Grant Office appeal form. [GO-12] [GO-08]
Chunks: ['GO-05', 'GO-08', 'GO-12', 'GO-06', 'GO-11', 'GO-13', 'KZ-14', 'GO-07']

SYSTEM: Embedding RAG
ANSWER: You have **14 calendar days from t

### 9. Is a vector database required to build a RAG system?

No, a vector database is not required. We can use BM25 or store embeddings in memory. A vector database is useful for large datasets.

### 10. Which retrieval approach worked better for this corpus, and what evidence supports that?

Embedding retrieval worked better in our experiment.

- BM25 Hit@3: 6/10 = 60%
- Embedding Hit@3: 10/10 = 100%

Embeddings found relevant documents more often because they understand similar meanings, even when the words are different.

C2 agreement counts:
- BM25 RAG: [enter result]/6
- Embedding RAG: [enter result]/6

### Conclusion

In this homework, I compared BM25 and EmbeddingGemma for document retrieval. I also tested how retrieval helps an LLM answer questions.

BM25 works well with exact words, while embeddings work better with similar meanings.

In my experiment, BM25 achieved 60% Hit@3 and EmbeddingGemma achieved 100% Hit@3.

RAG helped the model give more accurate answers using information from the documents. I learned that good retrieval is important for a reliable RAG system.
>